In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import psycopg2
import joblib


In [2]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/8.9 MB 2.6 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/8.9 MB 2.0 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.9 MB 2.1 MB/s eta 0:00:04
   ---------- ----------------------------- 2.4/8.9 MB 2.2 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/8.9 MB 2.3 MB/s eta 0:00:03
   --------------- ------------------------ 3.4/8.9 MB 2.4 MB/s eta 0:00:03
   ----------------- ---------------------- 3.9/8.9 MB 2.3 MB/s eta 0:00:03
   -------------------- ------------------- 4.5/8.9 MB 2.3 MB/s eta 0:00:02
   ---------------------- ----------------- 5.0/8.9 MB 2.4 MB/s eta 0:00:02
   ------------------------ --------------- 5.5/8.9 MB 2.4 MB/s eta 0:00:02
   --------------------------- ------------ 6.0/8.9 MB 2.4 MB/s eta 0:00:02
   -----------------------

In [8]:
conn = psycopg2.connect(
    host="localhost",
    database="ecopackai",
    user="postgres",
    password="1234"
)

query = """
SELECT
    strength_mpa,
    biodegradability_score,
    recyclability_percent,
    co2_emission_score,
    cost_per_unit,
    flexibility
FROM materials;
"""


df = pd.read_sql(query, conn)
conn.close()

df.head()



C:\Users\VARSHITHA\AppData\Local\Temp\ipykernel_23628\1333687404.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,strength_mpa,biodegradability_score,recyclability_percent,co2_emission_score,cost_per_unit,flexibility
0,21,4,41,3.20,5,Low
1,15,3,91,1.46,32,High
2,10,3,90,3.69,21,High
3,25,1,68,7.35,18,Medium
4,13,7,80,5.62,12,Medium


In [14]:
print(df.columns)
print(df["flexibility"].unique())


Index(['strength_mpa', 'biodegradability_score', 'recyclability_percent',
       'co2_emission_score', 'cost_per_unit', 'flexibility'],
      dtype='object')
['Low' 'High' 'Medium']


In [31]:
joblib.dump(rf_cost, "../models/cost_model.pkl")
joblib.dump(rf_co2, "../models/co2_model.pkl")


['../models/co2_model.pkl']

In [1]:
flexibility_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

flexibility_value = data["flexibility"]

if isinstance(flexibility_value, str):
    flexibility_encoded = flexibility_map.get(flexibility_value)
else:
    flexibility_encoded = int(flexibility_value)

if flexibility_encoded is None:
    return jsonify({
        "error": "flexibility must be one of: Low, Medium, High"
    }), 400


NameError: name 'data' is not defined

In [16]:
assert "flexibility_encoded" in df.columns
print(df[["flexibility", "flexibility_encoded"]].head())


  flexibility  flexibility_encoded
0         Low                    1
1        High                    3
2        High                    3
3      Medium                    2
4      Medium                    2


In [17]:
X_cost = df[
    [
        "strength_mpa",
        "biodegradability_score",
        "recyclability_percent",
        "co2_emission_score",
        "flexibility_encoded"
    ]
]

y_cost = df["cost_per_unit"]


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cost,
    y_cost,
    test_size=0.2,
    random_state=42
)


In [19]:
from sklearn.ensemble import RandomForestRegressor

rf_cost = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_cost.fit(X_train, y_train)


,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred = rf_cost.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R² Score:", r2)


RMSE: 14.329730109112313
R² Score: 0.07902240312163611


In [21]:
feature_importance = pd.Series(
    rf_cost.feature_importances_,
    index=X_cost.columns
).sort_values(ascending=False)

feature_importance


recyclability_percent     0.261208
co2_emission_score        0.255356
strength_mpa              0.228889
biodegradability_score    0.181170
flexibility_encoded       0.073377
dtype: float64

In [22]:
import joblib

joblib.dump(rf_cost, "ecopackai_cost_model.pkl")


['ecopackai_cost_model.pkl']

In [23]:
X_co2 = df[
    [
        "strength_mpa",
        "biodegradability_score",
        "recyclability_percent",
        "cost_per_unit",
        "flexibility_encoded"
    ]
]

y_co2 = df["co2_emission_score"]


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_co2,
    y_co2,
    test_size=0.2,
    random_state=42
)


In [25]:
from sklearn.ensemble import RandomForestRegressor

rf_co2 = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_co2.fit(X_train, y_train)


,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred = rf_co2.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("CO₂ Model RMSE:", rmse)
print("CO₂ Model R²:", r2)


CO₂ Model RMSE: 1.729998935043025
CO₂ Model R²: -0.18023815974660873


In [27]:
feature_importance_co2 = pd.Series(
    rf_co2.feature_importances_,
    index=X_co2.columns
).sort_values(ascending=False)

feature_importance_co2


recyclability_percent     0.323429
strength_mpa              0.307823
cost_per_unit             0.170416
biodegradability_score    0.145339
flexibility_encoded       0.052992
dtype: float64

In [30]:
import joblib

joblib.dump(rf_co2, "co2_model.pkl")


['co2_model.pkl']